In [4]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import shutil
from tqdm import tqdm, trange

from pathlib import Path
import ants
from nilearn.image import resample_to_img, load_img

plt.rcParams['axes.grid'] = False

plt.rcParams['font.family'] = 'Arial'

In [15]:
PLOT_PARAMS = dict(bbox_inches='tight', transparent=True, dpi=600)

PLOT_PATH = Path('../plots')
DATA_PATH = Path('../data')
RESULTS_PATH = Path('../results')


# Copy Internal Validity Index File

In [5]:
data_folder = Path('/home/guoqiu/NAS/Dep/Connectome_based_parcellation/results/')

file_name_mapping = {
    'Primate': '/home/guoqiu/NAS/Dep/Primate/results/primate/individual/internal_validity.tsv',
    'BCP_age_0_3_resting': data_folder / 'BCP/internal_validity.tsv',
    'dHCP_PCW_20_45_resting': data_folder / 'dHCP/internal_validity.tsv',
    'HCP_D_age_05_11_resting': data_folder / 'HCP_D/HCP_D_143/internal_validity.tsv',
    'HCP_D_age_12_21_resting': data_folder / 'HCP_D/HCP_D_144/internal_validity.tsv',
    'HCP_YA_age_22_35_resting': data_folder / 'HCP_YA/REST/internal_validity.tsv',
    'HCP_YA_age_22_35_DTI': data_folder / 'HCP_YA/DTI/internal_validity.tsv',
    'HCP_YA_age_22_35_movie': data_folder / 'HCP_7T/New_MOVIE/HCP_7T_MOVIE_clips_n172/internal_validity.tsv',
    'HCP_YA_age_22_35_7T_resting': data_folder / 'HCP_7T/REST/internal_validity.tsv',
    'HCP_A_age_36_60_resting': data_folder / 'HCP_A/HCP_A_720/internal_validity.tsv',
    'HCP_A_age_61_100_resting': data_folder / 'HCP_A/HCP_A_721/internal_validity.tsv',
    'Inhouse_18_30_resting': data_folder / 'Wanglab_VMPFC_Dataset/REST/internal_validity.tsv',
    'Inhouse_18_30_DTI': data_folder / 'Wanglab_VMPFC_Dataset/DTI/internal_validity.tsv',

}

# for name, from_file in file_name_mapping.items():
#     to_file = RESULTS_PATH / f'internal_validity/{name}.tsv'
#     shutil.copy(from_file, to_file)

# Copy and Transform Nifti File

## dHCP

In [9]:
top_path = Path("/home/guoqiu/NAS/HCPd/CBPtool/dHCP/results/test2/group")
assert top_path.exists(), 'DATABASE_PATH does not exist!'

# example input file:
# input_file = top_path / "3clusters/labeled_roi.nii.gz"
for input_file in top_path.rglob('*clusters/labeled_roi.nii.gz'):
    k_num = str(input_file.relative_to(top_path))[0]
    output_file = RESULTS_PATH / 'nii' / f'dHCP_PCW_20_45_resting_K{k_num}.nii.gz'

    # step1: baby template space
    baby_template_file = "/home/guoqiu/NAS/Baby_Brain/dHCP/Infant_registration_2step_extdhcp40wk/week40_T2w.nii.gz"
    baby_input_nib_img = resample_to_img(
        source_img=input_file,
        target_img=baby_template_file,
        interpolation='nearest')

    # step 2: baby template invert transform to MNI152
    baby_input_nib_img = ants.from_nibabel(baby_input_nib_img)
    MNI152_template_file = "/home/guoqiu/NAS/Baby_Brain/dHCP/Infant_registration_2step_extdhcp40wk/mni_icbm152_t1_tal_nlin_asym_09a_brain.nii"
    MNI152_template_ants_img = ants.image_read(MNI152_template_file)
    MNI152_affine = "/home/guoqiu/NAS/HCPd/CBPtool/dHCP/mask/mni2dhcp_invaffine.mat"
    MNI_input_ants_img = ants.apply_transforms(
        moving=baby_input_nib_img,
        fixed=MNI152_template_ants_img,
        whichtoinvert=[True],
        interpolator='nearestNeighbor',
        transformlist=[MNI152_affine])

    # step 3: MNI152 to vmpfc
    vmpfc_mask_file = "/home/guoqiu/NAS/HCPd/CBPtool/codes/k3_vmpfc_mask_binary.nii.gz"
    vmpfc_input_nib_img = resample_to_img(MNI_input_ants_img.to_nibabel(), vmpfc_mask_file, interpolation='nearest')

    vmpfc_input_nib_img.to_filename(output_file)
    print(f'from {input_file} \nto {output_file}')


from /home/guoqiu/NAS/HCPd/CBPtool/dHCP/results/test2/group/3clusters/labeled_roi.nii.gz 
to ../results/nii/dHCP_PCW_20_45_resting_K3.nii.gz
from /home/guoqiu/NAS/HCPd/CBPtool/dHCP/results/test2/group/6clusters/labeled_roi.nii.gz 
to ../results/nii/dHCP_PCW_20_45_resting_K6.nii.gz
from /home/guoqiu/NAS/HCPd/CBPtool/dHCP/results/test2/group/2clusters/labeled_roi.nii.gz 
to ../results/nii/dHCP_PCW_20_45_resting_K2.nii.gz
from /home/guoqiu/NAS/HCPd/CBPtool/dHCP/results/test2/group/5clusters/labeled_roi.nii.gz 
to ../results/nii/dHCP_PCW_20_45_resting_K5.nii.gz
from /home/guoqiu/NAS/HCPd/CBPtool/dHCP/results/test2/group/4clusters/labeled_roi.nii.gz 
to ../results/nii/dHCP_PCW_20_45_resting_K4.nii.gz


## BCP

In [13]:
top_path = Path('/home/guoqiu/NAS/Dep/Connectome_based_parcellation/results/BCP')
assert top_path.exists(), 'DATABASE_PATH does not exist!'
# example input file:
# input_file = top_path / "3clusters/labeled_roi.nii.gz"
for input_file in top_path.rglob('*clusters/labeled_roi.nii.gz'):
    k_num = str(input_file.relative_to(top_path))[0]
    output_file = RESULTS_PATH / 'nii' / f'BCP_age_0_3_resting_K{k_num}.nii.gz'

    # step1: baby template space
    baby_template_file = '/home/guoqiu/NAS/Dep/BCP/BCP_from_Helab/UNC-BCP_4D_Infant_Brain_Volumetric_Atlas/6Month/BCP-06M-T1.nii.gz'
    baby_input_nib_img = resample_to_img(
        source_img=input_file,
        target_img=baby_template_file,
        interpolation='nearest')

    # step 2: baby template invert transform to MNI152
    baby_input_nib_img = ants.from_nibabel(baby_input_nib_img)
    MNI152_template_file = "/home/guoqiu/NAS/Baby_Brain/dHCP/Infant_registration_2step_extdhcp40wk/mni_icbm152_t1_tal_nlin_asym_09a_brain.nii"
    MNI152_template_ants_img = ants.image_read(MNI152_template_file)
    transform_list = ants.registration(
        fixed=ants.image_read(baby_template_file),
        moving=MNI152_template_ants_img,
        type_of_transform='SyN')['invtransforms']
    MNI_input_ants_img = ants.apply_transforms(
        moving=baby_input_nib_img,
        fixed=MNI152_template_ants_img,
        interpolator='nearestNeighbor',
        transformlist=transform_list)

    # step 3: MNI152 to vmpfc
    vmpfc_mask_file = "/home/guoqiu/NAS/HCPd/CBPtool/codes/k3_vmpfc_mask_binary.nii.gz"
    vmpfc_input_nib_img = resample_to_img(MNI_input_ants_img.to_nibabel(), vmpfc_mask_file, interpolation='nearest')

    vmpfc_input_nib_img.to_filename(output_file)
    print(f'from {input_file} \nto {output_file}')


from /home/guoqiu/NAS/Dep/Connectome_based_parcellation/results/BCP/2clusters/labeled_roi.nii.gz 
to ../results/nii/BCP_age_0_3_resting_K2.nii.gz
from /home/guoqiu/NAS/Dep/Connectome_based_parcellation/results/BCP/3clusters/labeled_roi.nii.gz 
to ../results/nii/BCP_age_0_3_resting_K3.nii.gz
from /home/guoqiu/NAS/Dep/Connectome_based_parcellation/results/BCP/4clusters/labeled_roi.nii.gz 
to ../results/nii/BCP_age_0_3_resting_K4.nii.gz
from /home/guoqiu/NAS/Dep/Connectome_based_parcellation/results/BCP/5clusters/labeled_roi.nii.gz 
to ../results/nii/BCP_age_0_3_resting_K5.nii.gz
from /home/guoqiu/NAS/Dep/Connectome_based_parcellation/results/BCP/6clusters/labeled_roi.nii.gz 
to ../results/nii/BCP_age_0_3_resting_K6.nii.gz


## Others

In [ ]:
data_folder = Path('/home/guoqiu/NAS/Dep/Connectome_based_parcellation/results/')
to_folder = RESULTS_PATH / 'nii_by_label'
for cluster_k in trange(2, 7):
    file_name_mapping = {
        'HCP_D_age_05_11_resting': data_folder / f'HCP_D/HCP_D_143/{cluster_k}clusters/labeled_roi.nii.gz',
        'HCP_D_age_12_21_resting': data_folder / f'HCP_D/HCP_D_144/{cluster_k}clusters/labeled_roi.nii.gz',
        'HCP_YA_age_22_35_resting': data_folder / f'HCP_YA/REST/{cluster_k}clusters/labeled_roi.nii.gz',
        'HCP_A_age_36_60_resting': data_folder / f'HCP_A/HCP_A_720/{cluster_k}clusters/labeled_roi.nii.gz',
        'HCP_A_age_61_100_resting': data_folder / f'HCP_A/HCP_A_721/{cluster_k}clusters/labeled_roi.nii.gz',
        'HCP_YA_age_22_35_DTI': data_folder / f'HCP_YA/DTI/{cluster_k}clusters/labeled_roi.nii.gz',
        'HCP_YA_age_22_35_movie': data_folder / f'HCP_7T/New_MOVIE/HCP_7T_MOVIE_clips_n172/{cluster_k}clusters/labeled_roi.nii.gz',
        'Inhouse_18_30_resting': data_folder / f'Wanglab_VMPFC_Dataset/REST/{cluster_k}clusters/labeled_roi.nii.gz',
        'HCP_YA_age_22_35_7Tresting': data_folder / f'HCP_7T/REST/{cluster_k}clusters/labeled_roi.nii.gz',
        'Inhouse_18_30_DTI': data_folder / f'Wanglab_VMPFC_Dataset/DTI/{cluster_k}clusters/labeled_roi.nii.gz',
    }

    for name, from_file in file_name_mapping.items():
        to_file = RESULTS_PATH / f'nii/{name}_K{cluster_k}.nii.gz'
        shutil.copy(from_file, to_file)



# Nifti to Gifti

In [28]:
# Split Labels
to_folder = RESULTS_PATH / 'nii_by_label'
for from_file in (RESULTS_PATH / 'nii').iterdir():
    img = nib.load(from_file)
    img_data = np.round(img.get_fdata())

    cluster_k = int(from_file.stem.split('_K')[1][0])
    name = from_file.name.removesuffix('.nii.gz')
    for label_i in range(1, cluster_k + 1):
        label_img_data = (img_data == label_i).astype(int)
        label_img = nib.Nifti1Image(label_img_data, affine=img.affine, dtype=np.int8)
        nib.save(label_img, to_folder / f'{name}_K{cluster_k}_label{label_i}.nii.gz')

In [29]:
surf_file = "/home/guoqiu/guoqiu/Database/HCP_S1200_GroupAvg_v1/S1200.L.midthickness_MSMAll.32k_fs_LR.surf.gii"

from_file_list = list((RESULTS_PATH / 'nii_by_label').iterdir())
for from_file in tqdm(from_file_list):
    from_file_str = str(from_file.resolve())
    to_file_str = (from_file_str
                   .replace('nii_by_label', 'gii_by_label')
                   .replace('.nii.gz', '.func.gii'))
    shell_code = f'wb_command -volume-to-surface-mapping "{from_file_str}" "{surf_file}" "{to_file_str}" -enclosing'
    os.system(shell_code)

100%|█████████████████████████████████████████| 528/528 [01:36<00:00,  5.50it/s]
